# 🚀 RSNA Knee Abnormality Detection: PyTorch 3D ConvNeXt Pipeline

This notebook implements a **Full 3D ConvNeXt-Small Multi-View Architecture** for volumetric MRI classification:
1. **3D Volume Sampling:** Loads 16 uniformly sampled DICOM slices across Sagittal, Coronal, and Axial planes `(Shape: [3, 16, 192, 192])`.
2. **3D Spatial Augmentations:** Includes spatial flips and intensity variations on volumetric data.
3. **Custom 3D ConvNeXt:** Processes depth, height, and width simultaneously using 3D depthwise separable convolutions.
4. **Mixed Precision (AMP):** Optimized GPU training using `torch.cuda.amp`.

In [ ]:
import os
import gc
import math
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

class CFG:
    data_dir = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
    if not data_dir.exists():
        data_dir = Path('/kaggle/input/rsna-knee-abnormality-detection')
        
    model_name = 'convnext_small_3d'
    img_size = (192, 192)
    num_slices = 16         # Глибина 3D об'єму для кожного плану
    batch_size = 4
    epochs = 5
    lr = 3e-4
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 2
    use_amp = True          # Mixed Precision для прискорення 3D конволюцій

print(f"Using Device: {CFG.device}")
print(f"Data Directory: {CFG.data_dir}")

## 📁 1. Data Preparation & Splitting
Завантаження CSV-файлів, обробка пропущених міток та розділення на Train / Validation.

In [ ]:
train_df = pd.read_csv(CFG.data_dir / 'train.csv')
train_series_df = pd.read_csv(CFG.data_dir / 'train_series.csv')
test_df = pd.read_csv(CFG.data_dir / 'test.csv')

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

labeled_df = train_df.dropna(subset=TARGET_COLS, how='all').reset_index(drop=True)
labeled_df[TARGET_COLS] = labeled_df[TARGET_COLS].fillna(0).astype(np.float32)

val_size = int(len(labeled_df) * 0.2)
train_split = labeled_df.iloc[:-val_size].reset_index(drop=True)
val_split = labeled_df.iloc[-val_size:].reset_index(drop=True)

print(f"Train Split: {len(train_split):,} | Val Split: {len(val_split):,}")

## 🩺 2. 3D Multi-View Dataset & Augmentations
Зчитування серії DICOM-зрізів у 3D-тензор `(C, D, H, W)` та застосування 3D-аугментацій.

In [ ]:
class Knee3DMRIDataset(Dataset):
    def __init__(self, df, series_df, is_train=True, is_test=False):
        self.df = df
        self.series_df = series_df
        self.is_train = is_train
        self.is_test = is_test
        self.series_dir_name = 'test_series' if is_test else 'train_series'

    def __len__(self):
        return len(self.df)

    def _apply_3d_augmentations(self, volume):
        # 3D аугментації: відображення та масштабування інтенсивності
        if np.random.rand() > 0.5:
            volume = np.flip(volume, axis=-1)  # Горизонтальний flip
        if np.random.rand() > 0.5:
            volume = np.flip(volume, axis=-2)  # Вертикальний flip
        if np.random.rand() > 0.5:
            scale = np.random.uniform(0.8, 1.2)
            volume = np.clip(volume * scale, 0.0, 1.0)
        return volume.copy()

    def _load_3d_dicom_volume(self, study_uid, series_uid):
        """Зчитує 16 рівномірно розподілених зрізів -> (D, H, W)"""
        path = CFG.data_dir / self.series_dir_name / str(study_uid) / str(series_uid)
        dcm_files = sorted(list(path.glob('*.dcm')))
        
        if not dcm_files:
            return np.zeros((CFG.num_slices, CFG.img_size[0], CFG.img_size[1]), dtype=np.float32)
        
        # Рівномірне вибіркове зчитування 16 зрізів по всій глибині MRI
        indices = np.linspace(0, len(dcm_files) - 1, CFG.num_slices, dtype=int)
        
        slices = []
        for idx in indices:
            try:
                dcm = pydicom.dcmread(dcm_files[idx])
                img = dcm.pixel_array.astype(np.float32)
                try:
                    img = apply_voi_lut(img, dcm)
                except Exception:
                    pass
                
                img_min, img_max = img.min(), img.max()
                if img_max > img_min:
                    img = (img - img_min) / (img_max - img_min)
                else:
                    img = np.zeros_like(img)
                
                img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0)
                img_resized = torch.nn.functional.interpolate(
                    img_tensor, size=CFG.img_size, mode='bilinear', align_corners=False
                ).squeeze()
                
                slices.append(img_resized.numpy())
            except Exception:
                slices.append(np.zeros(CFG.img_size, dtype=np.float32))
                
        vol = np.stack(slices, axis=0) # (16, H, W)
        if self.is_train:
            vol = self._apply_3d_augmentations(vol)
        return vol

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = row['StudyInstanceUID']
        study_series = self.series_df[self.series_df['StudyInstanceUID'] == study_uid]
        
        planes_data = []
        for plane in ['Sagittal', 'Coronal', 'Axial']:
            plane_series = study_series[study_series['Anatomical_Plane'] == plane]
            if len(plane_series) > 0:
                s_uid = plane_series.iloc[0]['SeriesInstanceUID']
                vol = self._load_3d_dicom_volume(study_uid, s_uid)
            else:
                vol = np.zeros((CFG.num_slices, CFG.img_size[0], CFG.img_size[1]), dtype=np.float32)
            
            planes_data.append(torch.tensor(vol, dtype=torch.float32).unsqueeze(0))
            
        # Об'єднуємо 3 площини у канали -> (3, D, H, W)
        multi_plane_3d = torch.cat(planes_data, dim=0)
        
        if self.is_test:
            return multi_plane_3d, study_uid
            
        labels = torch.tensor(row[TARGET_COLS].values.astype(np.float32))
        return multi_plane_3d, labels

## 🏗️ 3. 3D ConvNeXt-Small Architecture
Реалізація блоків `Conv3d` та `LayerNorm` для обробки об'ємних медичних сканів.

In [ ]:
class ConvNeXtBlock3D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv3d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 4, 1) # (N, C, D, H, W) -> (N, D, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = x.permute(0, 4, 1, 2, 3) # Назад до (N, C, D, H, W)
        return residual + x

class ConvNeXtSmall3D(nn.Module):
    def __init__(self, in_channels=3, num_classes=len(TARGET_COLS)):
        super().__init__()
        # Stem Layer
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, 48, kernel_size=(3, 4, 4), stride=(2, 4, 4), padding=(1, 0, 0)),
            nn.GroupNorm(1, 48)
        )
        
        # Stages
        self.stage1 = nn.Sequential(ConvNeXtBlock3D(48), ConvNeXtBlock3D(48))
        self.downsample1 = nn.Sequential(nn.Conv3d(48, 96, kernel_size=2, stride=2), nn.GroupNorm(1, 96))
        
        self.stage2 = nn.Sequential(ConvNeXtBlock3D(96), ConvNeXtBlock3D(96))
        self.downsample2 = nn.Sequential(nn.Sequential(nn.Conv3d(96, 192, kernel_size=2, stride=2), nn.GroupNorm(1, 192)))
        
        self.stage3 = nn.Sequential(ConvNeXtBlock3D(192), ConvNeXtBlock3D(192))
        
        # Global Pooling & Head
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(192, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (Batch, 3_planes, 16_slices, H, W)
        x = self.stem(x)
        x = self.stage1(x)
        x = self.downsample1(x)
        x = self.stage2(x)
        x = self.downsample2(x)
        x = self.stage3(x)
        
        feat = self.pool(x).flatten(1)
        logits = self.classifier(feat)
        return logits

# Перевірка розмірностей тензора
model_test = ConvNeXtSmall3D()
dummy_input = torch.randn(2, 3, 16, 192, 192)
out = model_test(dummy_input)
print(f"✅ Output Logits Shape: {out.shape} (Batch Size x {len(TARGET_COLS)} Targets)")

## 🔁 4. Model Training with Mixed Precision (AMP)
Навчання моделі з використанням `torch.cuda.amp.GradScaler` для збереження пам'яті GPU при 3D-конволюціях.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Сучасний синтаксис torch.amp.autocast
        with torch.amp.autocast('cuda', enabled=CFG.use_amp):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        
    return running_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validation", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Сучасний синтаксис torch.amp.autocast
            with torch.amp.autocast('cuda', enabled=CFG.use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(labels.cpu().numpy())
            
    return running_loss / len(dataloader.dataset), np.vstack(all_preds), np.vstack(all_targets)

# Підготовка DataLoader
train_dataset = Knee3DMRIDataset(train_split, train_series_df, is_train=True)
val_dataset = Knee3DMRIDataset(val_split, train_series_df, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

model = ConvNeXtSmall3D().to(CFG.device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-4)

# Сучасний синтаксис torch.amp.GradScaler
scaler = torch.amp.GradScaler('cuda', enabled=CFG.use_amp)

print("🚀 Starting 3D ConvNeXt Training...")
best_val_loss = float('inf')

for epoch in range(1, CFG.epochs + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, CFG.device)
    val_loss, val_preds, val_targets = validate(model, val_loader, criterion, CFG.device)
    
    print(f"Epoch [{epoch}/{CFG.epochs}] | Train BCE: {train_loss:.4f} | Val BCE: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_knee_3d_convnext.pth')
        print("   -> Model Saved!")

## 🔮 5. Inference & Submission Generation
Завантаження кращих вагок, розрахунок передбачень для тестового набору та збереження `submission.csv`.

In [ ]:
test_series_df = pd.read_csv(CFG.data_dir / 'test_series.csv')
test_dataset = Knee3DMRIDataset(test_df, test_series_df, is_train=False, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

# Завантаження збережених вагок
model.load_state_dict(torch.load('best_knee_3d_convnext.pth', map_location=CFG.device))
model.eval()

submission_preds, study_uids = [], []

print("📦 Running 3D Test Set Inference...")
with torch.no_grad():
    for inputs, uids in tqdm(test_loader, desc="Inference"):
        inputs = inputs.to(CFG.device)
        with torch.amp.autocast('cuda', enabled=CFG.use_amp):
            outputs = model(inputs)
        
        # Перетворимо в float32, щоб уникнути overflow warnings у pandas
        probs = torch.sigmoid(outputs).to(torch.float32).cpu().numpy()
        submission_preds.append(probs)
        study_uids.extend(uids)

submission_preds = np.vstack(submission_preds)
sub_df = pd.DataFrame({'StudyInstanceUID': study_uids})
for i, col in enumerate(TARGET_COLS):
    sub_df[col] = submission_preds[:, i]

sub_df.to_csv('submission.csv', index=False)
print("\n✅ Successfully generated submission.csv!")
print(sub_df.head())